# Notebook 23 — In-silico perturbation screens with PSF

PSF's v0.6 Phase 2 `pathway_subtyping.perturb` layer lets you ask
counterfactual questions the observed-data pipeline cannot: "what would
the molecular-state vector (MSV) look like if this gene were knocked
out?" The answer is produced by:

1. `GeneformerPerturber` — perturbs a cell's embedding via the
   Geneformer foundation model (or a deterministic PCA fallback for
   local testing).
2. `MSVFromEmbedding` — a small learned head mapping embeddings to
   PSF pathway scores.
3. `PerturbationScreen` — runs a panel of gene knockouts and ranks
   them by impact on the MSV.
4. (Optional) `ConformalPathwayPredictor` (F1) — wraps the chain in
   calibrated prediction intervals.

Research use only. Not for clinical decision-making.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathway_subtyping.perturb import (
    FallbackPerturber,
    GeneformerPerturber,
    MSVFromEmbedding,
    PerturbationMode,
    PerturbationReport,
    PerturbationScreen,
)

rng = np.random.default_rng(42)

## 1. Synthetic cohort with an identifiable master regulator

Three cell clusters, each driven by a distinct marker gene. `MARKER_A`
is the master regulator of cluster 0's `pathway_0` by construction; the
same pattern holds for MARKER_B/pathway_1 and MARKER_C/pathway_2.

Swap in your own expression matrix + pathway scores when running on
real data.

In [ ]:
n_per = 100
n_genes = 50
cluster_profiles = np.zeros((3, n_genes))
cluster_profiles[0, 0] = 5.0  # MARKER_A
cluster_profiles[1, 1] = 5.0  # MARKER_B
cluster_profiles[2, 2] = 5.0  # MARKER_C

cells = []
for c in range(3):
    noise = rng.normal(0, 1.0, size=(n_per, n_genes))
    cells.append(cluster_profiles[c] + noise)
X = np.vstack(cells)

gene_names = ['MARKER_A', 'MARKER_B', 'MARKER_C'] + [f'GENE_{i}' for i in range(3, n_genes)]
expression = pd.DataFrame(X, columns=gene_names)

pathway_scores = pd.DataFrame({
    'pathway_0': X[:, 0],
    'pathway_1': X[:, 1],
    'pathway_2': X[:, 2],
    'pathway_generic': X[:, 3:10].mean(axis=1),
})
expression.shape, pathway_scores.shape

## 2. Fit the MSV head

In production the head is trained during package build. Here we fit
it on the synthetic reference so the downstream screen has something
to translate Geneformer embeddings into.

In [ ]:
perturber = GeneformerPerturber(FallbackPerturber(embedding_dim=16))
baseline_embedding = perturber.embed(expression)
head = MSVFromEmbedding().fit(baseline_embedding, pathway_scores)
head.pathway_names

## 3. Run a perturbation screen

Pass the screen a panel of gene names. Each one is independently
knocked out, the perturbed embedding is translated to MSV space, and
the per-gene delta-MSV row is recorded.

In [ ]:
screen = PerturbationScreen(perturber, head)
result = screen.run(
    expression,
    gene_panel=['MARKER_A', 'MARKER_B', 'MARKER_C', 'GENE_10', 'GENE_20'],
    mode=PerturbationMode.KNOCKOUT,
)
result.rank()

## 4. Directional signature check

The roadmap acceptance criterion for this layer is that knocking out a
known master regulator produces the directionally-expected MSV shift.
We verify that here by asking for expected signs per (gene, pathway).

In [ ]:
report = PerturbationReport.from_screen(result)
print(report.summary())

signature = {
    'MARKER_A': {'pathway_0': -1},
    'MARKER_B': {'pathway_1': -1},
    'MARKER_C': {'pathway_2': -1},
}
report.check_directional_signature(signature)

In [ ]:
fig = report.plot_top_k(5)
plt.show()

## 5. Conformal intervals on perturbed MSV (F1 integration)

Wrap the `MSVFromEmbedding` head with a `ConformalPathwayPredictor`
to get calibrated intervals on perturbed-MSV predictions. The same
protocol demonstrated in Notebook 21 applies unchanged.

In [ ]:
from pathway_subtyping.uncertainty import ConformalPathwayPredictor

def score_fn(emb):
    return head.transform(emb)['pathway_0'].to_numpy()

perm = rng.permutation(len(baseline_embedding))
cal_idx = perm[:len(perm)//2]
te_idx = perm[len(perm)//2:]

predictor = ConformalPathwayPredictor(score_fn=score_fn, coverage=0.9)
predictor.calibrate(baseline_embedding[cal_idx], pathway_scores['pathway_0'].to_numpy()[cal_idx])
print(f'Empirical coverage: {predictor.coverage_on(baseline_embedding[te_idx], pathway_scores["pathway_0"].to_numpy()[te_idx]):.3f}')

## Further reading

- Theodoris CV et al. (2023). *Transfer learning enables predictions
  in network biology.* Nature 618:616-624. Geneformer foundation
  model used as the production backend.
- PSF v0.6 roadmap — Phase 2 F5:
  [docs/roadmap-v06-codeberg.md](../../docs/roadmap-v06-codeberg.md)
- API guide: [docs/guides/perturbation.md](../../docs/guides/perturbation.md)